# Automated Sales Outreach Agents

We will use two approaches-
* Orchestrating Agents with Code
* Orchestrating Agents with LLM - Tools
* Orchestrating Agents with LLM - Handoffs

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td>
            <h2 style="color:#ff7800;">IMPORTANT PLEASE READ - Sending Emails</h2>
            <span style="color:#ff7800;">We are going to write an agent that sends emails. The best way to do this is with an email provider
            like SendGrid or Resend. But this requires a fair amount of heavy-lifting: you need to send from a proper email host where you own the domain and you can prove it with DNS records. It's quite a hassle.<br/><br/>
            So we're going to do it the free & simple way: using SMTP server configuration to send direct from your email. It's an easy setup, but it's not as powerful (eg you can't receive emails). If you'd like to take this further, use SendGrid or Resend instead..<br/><br/>
            And actually the email send is optional. We're doing this to demonstrate an Agent sending emails. The important part is the Agent work, not the email sending. Feel free to swap out the function for a Pushover push if you'd prefer.
            </span>
        </td>
    </tr>
</table>

## Setting up to send emails from your SMTP server

### STEP 1: Determining your SMTP Server

Either google or ask ChatGPT / Claude for the SMTP server for your email. Here are some common ones. Some email providers might not have SMTP servers enabled (eg Microsoft 365 for work/school).

Google: smtp.gmail.com  
Outlook.com / Hotmail / Live: smtp-mail.outlook.com  
Microsoft 365: smtp.office365.com  
iCloud Mail: smtp.mail.me.com  

Add to your .env file:  

`EMAIL_SMTP_SERVER=xxxx`

### STEP 2: Obtain an app specific password

Google how to do this for your email provider. For gmail, you need to have 2-step verification on. Then visit this page:

https://myaccount.google.com/apppasswords

Give it any name; copy the password and add it to your .env file, removing the spaces that it adds. (It should be 16 characters, no spaces.)

`EMAIL_APP_PASSWORD=xxxx`

### STEP 3: Add in your email address:

`EMAIL_ADDRESS=xxx`

Remember to Save the .env file!

In [11]:
# Environment
from dotenv import load_dotenv

# Agents
from agents import Agent, Runner, trace, function_tool, ModelSettings
from agents.extensions.visualization import draw_graph
from openai.types.responses import ResponseTextDeltaEvent

# Others
import os
import asyncio
import smtplib
import requests
from email.message import EmailMessage

In [2]:
load_dotenv(override = True)

MODEL_NAME = "gpt-5.4-mini"

In [44]:
EMAIL_ADDRESS = os.getenv("SMTP_EMAIL")
EMAIL_APP_PASSWORD = os.getenv("SMTP_TOKEN")
EMAIL_SMTP_SERVER = os.getenv("SMTP_SERVER")

if EMAIL_ADDRESS:
    print("Email address is set")
else:
    print("Email address is not set")

if EMAIL_SMTP_SERVER:
    print("SMTP server is set")
else:
    print("SMTP server is not set")

if EMAIL_APP_PASSWORD:
    print("App password is set")
else:
    print("App password is not set")

USE_EMAIL = EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD

if USE_EMAIL:
    print("Email is set up and we will try using it")
else:
    print("Email is not set up; we will send push notifications instead")

Email address is set
SMTP server is set
App password is set
Email is set up and we will try using it


In [4]:
def send_email(subject: str, text_body: str, html_body:str):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Body"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype = "html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

In [5]:
send_email("Testing Testing 123", "Fingers crossed...", "<html><body><strong>Fingers</strong> crossed..</body></html>")

In [ ]:
# USE_EMAIL = False

## Alternative

In [7]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [8]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [9]:
def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

In [12]:
send_message("Big news", "Communications are a go!", "<html><body>Communications are a <strong>go!</strong></body></html>")

Push: Subject: Big news

Communications are a go!


## Agent Orchestration

there are two method for orchestration
* Orchestration by code: Predictatble and Deterministic 
* Orchestration by LLMs: More Powerful

https://openai.github.io/openai-agents-python/multi_agent/

__In simple terms__ - orchestration is basically when you make multiple calls to Runner.run(). Multiple calls to send  over to LLM nad then based on response from that you construct next call to LLM/Runner.run()

It is basically what we did in week one when we called LLM one after another, and it tends to be more reliable, simpler and the more bulletproof way of doing it. {talking about the Difficult question passed through different LLMs and ranked the answers/questions}

### 1. Orchestration by Code

In [13]:
intro = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails.
"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility."
instructions2 = intro + "Your email style is witty, engaging, and humorous."
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

In [14]:
sales_agent1 = Agent(name = "Professional Sales Agent", instructions = instructions1, model = MODEL_NAME)
sales_agent2 = Agent(name = "Humorous Sales Agent", instructions = instructions2, model = MODEL_NAME)
sales_agent3 = Agent(name = "Stylish Sales Agent", instructions = instructions3, model = MODEL_NAME)

In [18]:
result = Runner.run_streamed(sales_agent1, input = "Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end = "", flush = True)

Subject: Simplifying SOC 2 readiness without adding headcount

Hi [First Name],

I’m reaching out from ComplAI. We help teams streamline SOC 2 compliance and audit preparation with an AI-powered platform designed to reduce manual work, keep evidence organized, and make readiness easier to manage.

For many companies, SOC 2 becomes a distracting operational burden—especially when ownership is spread across security, engineering, and operations. ComplAI helps bring that process into one place so teams can move faster with more confidence.

If this is on your roadmap, I’d welcome the chance to share how we help teams prepare efficiently and stay audit-ready.

Would you be open to a brief conversation next week?

Best,  
[Your Name]  
ComplAI

In [26]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )

outputs = [result.final_output for result in results]

for index, output in enumerate(outputs):
    print(f"Email {index+1}: \n{output} \n\n")

Email 1: 
Subject: Simplifying SOC 2 readiness without slowing your team

Hi [First Name],

I’m reaching out from ComplAI. We help teams streamline SOC 2 preparation and stay audit-ready with an AI-powered platform built to reduce manual work and uncertainty.

For many growing companies, compliance becomes a distraction from product and sales. ComplAI helps centralize evidence, track controls, and prepare for audit with far less back-and-forth.

If SOC 2 is on your roadmap this quarter, I’d be glad to share how teams are using ComplAI to move faster with more confidence.

Open to a brief conversation next week?

Best,  
[Your Name]  
ComplAI 


Email 2: 
Subject: SOC 2, but make it less of a circus

Hi [First Name],

Hope your week’s going well—and not currently being held together by spreadsheets, screenshots, and sheer determination.

I’m reaching out from ComplAI, where we help teams get SOC 2 ready without turning compliance into a full-time personality trait. Our AI-powered platfo

In [27]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give an explanation; reply with the selected email only.
"""

sales_picker = Agent(name="Sales_picker", instructions=decision, model=MODEL_NAME)


In [30]:
message = "write a cold sales email"

with trace("Sales selection workflow"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

    outputs = [result.final_output for result in results]

    emails = f"cold sales email:\n\n \n\nEmail:\n\n{outputs}"

    best = await Runner.run(sales_picker, emails)

    print(f"The best sales email: \n\n {best.final_output}")

The best sales email: 

 Subject: Faster SOC 2 readiness with AI

Hi [First Name],

Teams preparing for SOC 2 usually run into the same issues: scattered evidence, manual follow-ups, and audit prep that takes too long.

ComplAI helps companies stay audit-ready by automating evidence collection, control tracking, and compliance workflows with AI.

If SOC 2 is on your roadmap, I’d be happy to show you how teams are using ComplAI to cut the prep burden and stay organized.

Open to a quick 15-minute intro next week?

Best,  
[Your Name]  
ComplAI


#### Now we add the tools: Email sender

In [38]:
@function_tool
def send_emails_tool(subject: str, text_body: str, html_body:str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

#### This has automatically been converted into a tool, with the boilerplate json created

In [35]:
send_emails_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_emails_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [41]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email.
"""

# This forces the agent to use the tool
require_tool = ModelSettings(tool_choice = "required")

sales_sender = Agent(name = "Sales Sender", instructions = decision, model = MODEL_NAME, tools = [send_emails_tool], model_settings = require_tool)

In [45]:
message = "Write a cold sales email"

with trace("Sales selection workflow with sending"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    response = await Runner.run(sales_sender, emails)

    print(f"Final response:\n{response.final_output}")

Final response:
Done — I chose and sent the “SOC 2 prep without the chaos” email.
